In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

(X_train, _), (X_test, _) = mnist.load_data()

X_train_np = X_train.reshape(X_train.shape[0], 784).astype('float32') / 255.0
X_test_np = X_test.reshape(X_test.shape[0], 784).astype('float32') / 255.0

print("Training data shape:", X_train_np.shape)
print("Testing data shape:", X_test_np.shape)

# PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

pca = PCA(n_components=32, random_state=42)

X_train_pca = pca.fit_transform(X_train_np)
X_test_pca = pca.transform(X_test_np)

X_test_pca_recon = pca.inverse_transform(X_test_pca)

pca_mse = mean_squared_error(X_test_np, X_test_pca_recon)

print("PCA Components:", pca.n_components_)
print("PCA Reconstruction MSE:", pca_mse)

# Autoencoder

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.utils.set_random_seed(42)

autoencoder = Sequential([
    Dense(256, activation="relu", input_shape=(784,)),
    Dense(128, activation="relu"),
    Dense(32, activation="linear", name="bottleneck_latent"),
    
    Dense(128, activation="relu"),
    Dense(256, activation="relu"),
    Dense(784, activation="linear")
])

autoencoder.compile(optimizer="adam", loss="mse")

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

print("Training Autoencoder...")
history = autoencoder.fit(
    X_train_np, X_train_np,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

X_test_ae_recon = autoencoder.predict(X_test_np)

ae_mse = mean_squared_error(X_test_np, X_test_ae_recon)
print("Autoencoder Reconstruction MSE:", ae_mse)

In [ ]:
print("MSE Comparison")
print(f"PCA MSE:         {pca_mse:.6f}")
print(f"Autoencoder MSE: {ae_mse:.6f}")

In [ ]:
n_images = 8
plt.figure(figsize=(16, 6))

for i in range(n_images):
    ax = plt.subplot(3, n_images, i + 1)
    plt.imshow(X_test_np[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    if i == 0: ax.set_title("Original Image")

    ax = plt.subplot(3, n_images, i + 1 + n_images)
    plt.imshow(X_test_ae_recon[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    if i == 0: ax.set_title("Autoencoder")
        
    ax = plt.subplot(3, n_images, i + 1 + 2 * n_images)
    plt.imshow(X_test_pca_recon[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    if i == 0: ax.set_title("PCA")

plt.tight_layout()
plt.show()